In [3]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# =========================================================
# ΒΗΜΑ 5: Gaussian Naïve Bayes (From Scratch)
# =========================================================

# 1. Προετοιμασία
data = load_wine()
X = data.data
y = data.target

# Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Σημείωση: Στον Naive Bayes δεν είναι απαραίτητο το Scaling (StandardScaler),
# καθώς ο αλγόριθμος μαθαίνει τις παραμέτρους (mean, var) στη φυσική κλίμακα των δεδομένων.

# 2. Κλάση/Συνάρτηση Εκπαίδευσης (Fit)
def fit_naive_bayes(X, y):
    n_samples, n_features = X.shape
    classes = np.unique(y)
    n_classes = len(classes)

    # Αρχικοποίηση πινάκων για μέσους όρους, διακυμάνσεις και priors
    means = np.zeros((n_classes, n_features))
    vars = np.zeros((n_classes, n_features))
    priors = np.zeros(n_classes)

    # Υπολογισμός στατιστικών για κάθε κλάση
    for idx, c in enumerate(classes):
        # Παίρνουμε μόνο τα δεδομένα που ανήκουν στην κλάση c
        X_c = X[y == c]
        
        # Υπολογίζουμε mean και variance για κάθε feature
        means[idx, :] = X_c.mean(axis=0)
        vars[idx, :] = X_c.var(axis=0)
        
        # Prior πιθανότητα P(Class): Πόσο συχνή είναι η κλάση στο dataset
        priors[idx] = X_c.shape[0] / float(n_samples)

    return classes, means, vars, priors

# 3. Gaussian Probability Density Function (PDF)
def gaussian_pdf(class_idx, x, means, vars):
    """
    Υπολογίζει την πιθανότητα P(x|class) βασισμένη στον τύπο της κανονικής κατανομής.
    """
    mean = means[class_idx]
    var = vars[class_idx]
    
    # Προσθέτουμε ένα πολύ μικρό νούμερο (epsilon) στη διακύμανση για να αποφύγουμε διαίρεση με το 0
    epsilon = 1e-9
    numerator = np.exp(- (x - mean)**2 / (2 * (var + epsilon)))
    denominator = np.sqrt(2 * np.pi * (var + epsilon))
    
    return numerator / denominator

# 4. Συνάρτηση Πρόβλεψης (Predict)
def predict_naive_bayes(X, classes, means, vars, priors):
    y_pred = []

    for x in X:
        posteriors = []

        # Υπολογισμός πιθανότητας για κάθε κλάση
        for idx, c in enumerate(classes):
            # Prior: log(P(Class))
            prior = np.log(priors[idx])
            
            # Likelihood: sum(log(P(Feature|Class)))
            # Χρησιμοποιούμε log για να μετατρέψουμε τον πολλαπλασιασμό σε πρόσθεση (αποφυγή underflow)
            class_conditional = np.sum(np.log(gaussian_pdf(idx, x, means, vars)))
            
            # Posterior = Prior + Likelihood (σε λογαριθμική κλίμακα)
            posterior = prior + class_conditional
            posteriors.append(posterior)
        
        # Επιλογή της κλάσης με τη μέγιστη πιθανότητα
        y_pred.append(classes[np.argmax(posteriors)])

    return np.array(y_pred)

# --- ΕΚΤΕΛΕΣΗ ---

# Εκπαίδευση
classes, means, vars, priors = fit_naive_bayes(X_train, y_train)
print("Εκπαίδευση ολοκληρώθηκε.")
print(f"Βρέθηκαν {len(classes)} κλάσεις.")
print(f"Διαστάσεις Πίνακα Μέσων Τιμών: {means.shape} (Κλάσεις x Features)")

# Πρόβλεψη
y_pred_train = predict_naive_bayes(X_train, classes, means, vars, priors)
y_pred_test = predict_naive_bayes(X_test, classes, means, vars, priors)

# Αξιολόγηση
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print("\n=== Αποτελέσματα Gaussian Naïve Bayes ===")
print(f"Accuracy (Train): {acc_train * 100:.2f}%")
print(f"Accuracy (Test):  {acc_test * 100:.2f}%")

Εκπαίδευση ολοκληρώθηκε.
Βρέθηκαν 3 κλάσεις.
Διαστάσεις Πίνακα Μέσων Τιμών: (3, 13) (Κλάσεις x Features)

=== Αποτελέσματα Gaussian Naïve Bayes ===
Accuracy (Train): 97.18%
Accuracy (Test):  100.00%
